In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

True

In [2]:
from crewai_tools import SerperDevTool

# 도구 인스턴스 생성
search_tool = SerperDevTool()

In [6]:
from crewai import Agent, Task, Crew, Process

# 총괄 여행 플래너(관리자 Agent)
planner_agent = Agent(
    role="총괄 여행 플래너",
    goal="최적의 서울 근교 1박 2일 여행 일정과 추천 음식을 종합하여 최종 여행 계획서를 작성",
    backstory="10년 경력의 베테랑 여행 컨설턴트로 다양한 분야의 의견을 취합하여 여행 계획을 완성",
    allow_delegation=True,       # 다른 에이전트에게 업무 위임을 허용
    llm='gpt-5.4-mini',
    verbose=True
)

# 여행 전문가 Agent(여행 일정 추천 담당)
travel_agent = Agent(
    role="여행 전문가",
    goal="최신 여행 트렌드를 조사하여 서울 근교의 인기 있는 여행지와 일정을 제안",
    backstory="국내 여행지에 대해 잘 알고 있는 전문가로, 최근 여행 트렌드를 바탕으로 관광지 추천",
    tools=[search_tool],
    llm='gpt-5.4-mini',
    verbose=True
)

# 요리 전문가 Agent(음식 추천 담당)
culinary_agent = Agent(
    role="요리 전문가",
    goal="추천된 여행지와 잘 어울리는 현지 음식 및 레시피 추천",
    backstory="국내 각 지역의 음식 문화와 레시피에 능통한 전문가로, 여행지에 어울리는 음식을 추천",
    tools=[search_tool],
    llm='gpt-5.4-mini',
    verbose=True
)

In [7]:
# Task 정의 (총괄 플래너에게 최종 여행 계획서 작성 지시)
planner_task = Task(
    description=(
        "국내 최신 여행 트렌드가 반영된 서울 근교의 1박 2일 여행 일정을 작성하고,"
        "각 여행지와 잘 어울리는 현지 음식과 레시피를 포함하여 여행 계획서를 한국어로 작성해주세요."
    ),
    expected_output=(
        "최신 여행 트렌드를 반영한 서울 근교 1박 2일 여행 일정과"
        "각 여행지의 현지 음식 및 간단한 레시피를 포함한 한국어 여행 계획서"
    ),
    # agent=planner_agent
)

In [8]:
# Crew 구성 (계층적 프로세스 사용)
# 매니저 설정이 제일 중요.
crew = Crew(
    agents=[travel_agent, culinary_agent],  # 하위 실행에 참여할 에이전트들(관리자 제외)
    tasks=[planner_task],
    process=Process.hierarchical,       # 매니저한테 goal을 디테일하게 작성해야 됨.
    manager_agent=planner_agent,        # 총괄 여행 플래너를 매니저로 지정.
    verbose=True
)

In [9]:
result = await crew.kickoff_async()
print("\n\n📗 최종 여행 계획서:\n")
print(result)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: b2a2bbae-1f06-4bdc-8408-facf744d3323                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: 국내 최신 여행 트렌드가 반영된 서울 근교의 1박 2일 여행 일정을 작성하고,각 여행지와 잘 어울리는 현지     │
│  음식과 레시피를 포함하여 여행 계획서를 한국어로 작성해주세요.                                                  │
│  ID: 5597be55-cd3d-474e-a513-b32a1cc0ca42                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: 총괄 여행 플래너                                                                                        │
│                                                                                                                 │
│  Task: 국내 최신 여행 트렌드가 반영된 서울 근교의 1박 2일 여행 일정을 작성하고,각 여행지와 잘 어울리는 현지     │
│  음식과 레시피를 포함하여 여행 계획서를 한국어로 작성해주세요.                                                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: delegate_work_to_coworker                                                                                │
│  Args: {'task': '국내 최신 여행 트렌드를 반영한 서울 근교 1박 2일 여행 코스를 추천해 주세요. 최근 선호되는      │
│  감성 숙소, 로컬 체험, 힐링/자연, 사진 스팟, 대중교통 접근성, 혼잡도 분산 관점까지 고려해서 가장 적합한 지역    │
│  1곳과 구체적인 1박 2일 일정(아침~저녁)을 짜 주세요. 각 일정에 맞는 핵심 방문지와 이동 동선도 함께 알려         │
│  주세요.', '...                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#2) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: delegate_work_to_coworker                                                                                │
│  Args: {'task': '서울 근교 1박 2일 여행 일정과 잘 어울리는 현지 음식 4~6개를 추천해 주세요. 각 음식은 여행지    │
│  분위기와 연결되어야 하며, 집에서도 따라 하기 쉬운 간단 레시피를 3~6단계로 한국어로 작성해 주세요. 가능하면     │
│  지역의 대표 재료나 향토성을 반영해 주세요.', 'context': '사용자는 한국어로 된 최종 여행 계획서를 원합니다.     │
│  서울...                                                                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: 총괄 여행 플래너                                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # 서울 근교 1박 2일 여행 계획서                                                                                │
│  ## 최신 여행 트렌드 반영형: **양평 감성 힐링 여행**                                                            │
│                                                                                                                 │
│  요즘 국내 여행 트렌드는 단순한 관광보다                                                                        │
│  **감성 숙소, 로컬 체험, 자연 속 힐링, 사진이 잘 나오는 장소, 짧지만 밀도 높은 일정**이 핵심입니다.             │
│  이 기준에 가장 잘 맞는 서울 근교 여행지로 **양평**을 추천합니다.                                               │
│                                                                                                                 │
│  양평은                                                                                                         │
│  - 서울에서 비교적 가깝고                                                                                       │
│  - 대중교통 접근성이 좋으며                                                                                     │
│  - 두물머리, 세미원, 강변 산책길 같은 자연 명소가 있고                                                          │
│  - 감성 숙소와 로컬 식당, 카페가 풍부해                                                                         │
│  **1박 2일 여행 만족도가 매우 높은 지역**입니다.                                                                │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  # 1. 여행 콘셉트                                                                                               │
│  ## **“강과 정원, 숲과 감성 숙소를 함께 즐기는 힐링 여행”**                                                     │
│                                                                                                                 │
│  - 아침에는 강변 산책                                                                                           │
│  - 낮에는 정원과 로컬 카페                                                                                      │
│  - 저녁에는 감성 숙소에서 휴식                                                                                  │
│  - 다음 날에는 숲길과 로컬 음식으로 마무리                                                                      │
│                                                                                                                 │
│  빡빡하게 많이 보는 여행보다,                                                                                   │
│  **천천히 쉬고 먹고 걷는 여행**에 잘 맞는 일정입니다.                                                           │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  # 2. 추천 여행지 선정 이유: 왜 양평인가?                                                                       │
│                                                                                                                 │
│  양평은 최근 여행 트렌드인 **느린 여행, 자연 여행, 감성 숙소 여행**에 가장 잘 어울립니다.                       │
│                                                    

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 국내 최신 여행 트렌드가 반영된 서울 근교의 1박 2일 여행 일정을 작성하고,각 여행지와 잘 어울리는 현지     │
│  음식과 레시피를 포함하여 여행 계획서를 한국어로 작성해주세요.                                                  │
│  Agent: 총괄 여행 플래너                                                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: b2a2bbae-1f06-4bdc-8408-facf744d3323                                                                       │
│  Final Output: # 서울 근교 1박 2일 여행 계획서                                                                  │
│  ## 최신 여행 트렌드 반영형: **양평 감성 힐링 여행**                                                            │
│                                                                                                                 │
│  요즘 국내 여행 트렌드는 단순한 관광보다                                                                        │
│  **감성 숙소, 로컬 체험, 자연 속 힐링, 사진이 잘 나오는 장소, 짧지만 밀도 높은 일정**이 핵심입니다.             │
│  이 기준에 가장 잘 맞는 서울 근교 여행지로 **양평**을 추천합니다.                                               │
│                                                                                                                 │
│  양평은                                                                                                         │
│  - 서울에서 비교적 가깝고                                                                                       │
│  - 대중교통 접근성이 좋으며                                                                                     │
│  - 두물머리, 세미원, 강변 산책길 같은 자연 명소가 있고                                                          │
│  - 감성 숙소와 로컬 식당, 카페가 풍부해                                                                         │
│  **1박 2일 여행 만족도가 매우 높은 지역**입니다.                                                                │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  # 1. 여행 콘셉트                                                                                               │
│  ## **“강과 정원, 숲과 감성 숙소를 함께 즐기는 힐링 여행”**                                                     │
│                                                                                                                 │
│  - 아침에는 강변 산책                                                                                           │
│  - 낮에는 정원과 로컬 카페                                                                                      │
│  - 저녁에는 감성 숙소에서 휴식                                                                                  │
│  - 다음 날에는 숲길과 로컬 음식으로 마무리                                                                      │
│                                                                                                                 │
│  빡빡하게 많이 보는 여행보다,                                                                                   │
│  **천천히 쉬고 먹고 걷는 여행**에 잘 맞는 일정입니다.                                                           │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  # 2. 추천 여행지 선정 이유: 왜 양평인가?                                                                       │
│                                                                                                                 │
│  양평은 최근 여행 트렌드인 **느린 여행, 자연 여행, 감성 숙소 여행**에 가장 잘 어울립니다.                       │
│                                            



📗 최종 여행 계획서:

# 서울 근교 1박 2일 여행 계획서  
## 최신 여행 트렌드 반영형: **양평 감성 힐링 여행**

요즘 국내 여행 트렌드는 단순한 관광보다  
**감성 숙소, 로컬 체험, 자연 속 힐링, 사진이 잘 나오는 장소, 짧지만 밀도 높은 일정**이 핵심입니다.  
이 기준에 가장 잘 맞는 서울 근교 여행지로 **양평**을 추천합니다.

양평은  
- 서울에서 비교적 가깝고  
- 대중교통 접근성이 좋으며  
- 두물머리, 세미원, 강변 산책길 같은 자연 명소가 있고  
- 감성 숙소와 로컬 식당, 카페가 풍부해  
**1박 2일 여행 만족도가 매우 높은 지역**입니다.

---

# 1. 여행 콘셉트
## **“강과 정원, 숲과 감성 숙소를 함께 즐기는 힐링 여행”**

- 아침에는 강변 산책
- 낮에는 정원과 로컬 카페
- 저녁에는 감성 숙소에서 휴식
- 다음 날에는 숲길과 로컬 음식으로 마무리

빡빡하게 많이 보는 여행보다,  
**천천히 쉬고 먹고 걷는 여행**에 잘 맞는 일정입니다.

---

# 2. 추천 여행지 선정 이유: 왜 양평인가?

양평은 최근 여행 트렌드인 **느린 여행, 자연 여행, 감성 숙소 여행**에 가장 잘 어울립니다.

### 양평의 장점
- **서울 근교 1~2시간 내 접근 가능**
- **두물머리, 세미원**처럼 대표적인 사진 스팟이 있음
- **강변 산책 + 정원 + 카페 + 숙소 힐링** 구성이 가능
- **뚜벅이 여행도 가능**할 정도로 동선이 무리 없음
- 주말 혼잡을 피하면 **여유로운 여행**이 가능

---

# 3. 1박 2일 상세 일정

## Day 1. 강변 감성 + 정원 산책 + 숙소 힐링

### 08:00 서울 출발
- 서울역, 청량리역, 왕십리역 등에서 출발 가능
- 경의중앙선 또는 중앙선 계열 동선을 활용
- 가능하면 **오전 8시 전후 출발**이 좋습니다  
  → 두물머리 혼잡을 피할 수 있습니다.

---

### 09:30 두물머리 도착
## 핵심 방문지 1: **두물

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯